In [2]:
import pandas as pd
import networkx as nx
import time

# Carga de aristas y construccion del grafo
Esta celda mide el tiempo de carga de las aristas, crea el grafo con NetworkX y muestra un resumen rapido.

- Inicia un cronometro para estimar el tiempo total de carga.
- Lee `large_twitch_edges.csv` a un DataFrame.
- Convierte el DataFrame a un grafo `G` usando las columnas de origen y destino.
- Detiene el cronometro y reporta tiempo, nodos y aristas.

In [3]:
# 1. Iniciamos el cronómetro
inicio_carga = time.time() # Dado a que las aristas son 6 millones, el tiempo de carga puede ser significativo, por lo que hemos querido medirlo

# 2. Cargar los datos. 
print("Cargando datos...")
df = pd.read_csv('large_twitch_edges.csv') 

# 3. Convertir el DataFrame a un grafo de NetworkX
print("Creando grafo...")
G = nx.from_pandas_edgelist(df, source='numeric_id_1', target='numeric_id_2')

# 4. Paramos el cronómetro
fin_carga = time.time()

# 5. Mostramos los resultados
tiempo_total = fin_carga - inicio_carga
print("-----------------------------------")
print(f"Tiempo total: {tiempo_total:.2f} segundos")
print(f"Nodos cargados: {G.number_of_nodes()}")
print(f"Aristas cargadas: {G.number_of_edges()}")

Cargando datos...
Creando grafo...
-----------------------------------
Tiempo total: 11.43 segundos
Nodos cargados: 168114
Aristas cargadas: 6797557


# Subgrafo de 20k nodos
Aqui extraemos un subgrafo con los primeros 20,000 nodos para trabajar sobre un conjunto mas manejable.

- Usamos los ids 0..19999 para mantener un rango fijo y reproducible.
- Filtramos nodos inexistentes para evitar errores si faltan ids.
- Creamos `G_20k` y mostramos nodos y aristas para validar el tamanio.
- Convertimos a edgelist para inspeccionar la estructura y facilitar guardados posteriores.

In [4]:
def obtener_subgrafo_por_rango(G_base, inicio, fin):
    if inicio > fin:
        raise ValueError("'inicio' debe ser menor o igual que 'fin'.")
    # Filtra nodos inexistentes para evitar errores
    nodos = [n for n in range(inicio, fin + 1) if G_base.has_node(n)]
    return G_base.subgraph(nodos).copy()

# Subgrafo con los primeros 20k nodos (ids 0..19999)
G_20k = obtener_subgrafo_por_rango(G, 0, 19999)

print(f"Nodos en G_20k: {G_20k.number_of_nodes()}")
print(f"Aristas en G_20k: {G_20k.number_of_edges()}")

edges_20k = nx.to_pandas_edgelist(G_20k)
edges_20k.head()

#Subgrafo con 20k nodos a partir de 20k
G_20k_to_40k = obtener_subgrafo_por_rango(G, 20000, 39999)

print(f"Nodos en G_20k_to_40k: {G_20k_to_40k.number_of_nodes()}")
print(f"Aristas en G_20k_to_40k: {G_20k_to_40k.number_of_edges()}")

edges_20k_to_40k = nx.to_pandas_edgelist(G_20k_to_40k)
edges_20k_to_40k.head()

Nodos en G_20k: 20000
Aristas en G_20k: 92798
Nodos en G_20k_to_40k: 20000
Aristas en G_20k_to_40k: 98612


,source,target
0,20002,21991
1,20002,37579
2,20002,36975
3,20002,38022
4,20002,34793


# Centralidades y medidas relacionales (formulas)
En esta celda calculamos varias medidas de centralidad y estructura para cada nodo del subgrafo y las guardamos en un CSV. Esto enriquece las caracteristicas con informacion relacional que no esta en los atributos originales.

- **Centralidad de grado**: $C_D(v) = \frac{\deg(v)}{n - 1}$.

- **Intermediacion (betweenness)**: $C_B(v) = \sum_{s\neq v\neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$.

- **Cercania (closeness)**: $C_C(v) = \frac{n - 1}{\sum_{u\neq v} d(v, u)}$.

- **Clustering local**: $C(v) = \frac{2T(v)}{\deg(v)(\deg(v)-1)}$.

- **Triangulos**: $T(v)$ es el numero de triangulos que incluyen a $v$.

- **PageRank**: $PR(v) = \frac{1-d}{n} + d\sum_{u\in N^-(v)} \frac{PR(u)}{\deg(u)}$.

Donde $n$ es el numero de nodos, $\deg(v)$ es el grado de $v$ (numero de vecinos), $d(v, u)$ es la distancia geodesica entre $v$ y $u$, $T(v)$ es el numero de pares de vecinos de $v$ que estan conectados entre si (triangulos con $v$), $\sigma_{st}$ es el numero de caminos geodesicos entre $s$ y $t$, $\sigma_{st}(v)$ los que pasan por $v$, $PR(u)$ es el puntaje PageRank del nodo $u$, y $d$ es el factor de amortiguacion.

In [5]:
def centralidades_csv(G_sub, csv_path="datos_relacionales_20k.csv"):
    dc = nx.degree_centrality(G_sub)
    bc = nx.betweenness_centrality(G_sub, k=5000) # Para acelerar el cálculo, se puede usar una muestra de nodos para el cálculo de la centralidad de intermediación
    cc = nx.closeness_centrality(G_sub)
    clustering = nx.clustering(G_sub)
    triangles = nx.triangles(G_sub)
    pageRank = nx.pagerank(G_sub)

    df_out = pd.DataFrame({
        "nodo": list(G_sub.nodes()),
        "degree_centrality": [dc[n] for n in G_sub.nodes()],
        "betweenness_centrality": [bc[n] for n in G_sub.nodes()],
        "closeness_centrality": [cc[n] for n in G_sub.nodes()],
        "clustering": [clustering[n] for n in G_sub.nodes()],
        "triangles": [triangles[n] for n in G_sub.nodes()],
        "pageRank": [pageRank[n] for n in G_sub.nodes()]
    })
    df_out.to_csv(csv_path, index=False)
    return df_out

# centralidades_df = centralidades_csv(G_20k)
# centralidades_df.head()

centralidades_df_20k_to_40k = centralidades_csv(G_20k_to_40k, csv_path="datos_relacionales_20k_to_40k.csv")
centralidades_df_20k_to_40k.head()

,nodo,degree_centrality,betweenness_centrality,closeness_centrality,clustering,triangles,pageRank
0,20000,0.00000,0.000000e+00,0.000000,0.000000,0,0.000009
1,20001,0.00000,0.000000e+00,0.000000,0.000000,0,0.000009
2,20002,0.00035,2.177020e-04,0.240521,0.000000,0,0.000045
3,20003,0.00015,3.740329e-07,0.232296,0.000000,0,0.000019
4,20004,0.00045,4.174369e-04,0.289590,0.111111,4,0.000061


# Union de atributos y relaciones
En esta celda unificamos los atributos de `large_twitch_features.csv` con las medidas relacionales calculadas, alineando fila a fila para que el nodo 0 quede con la fila 0, etc. Esto produce un dataset final listo para modelado.

- Recorta ambos CSV a 20k filas y reinicia el indice para asegurar el alineamiento.
- Elimina la columna `nodo` de las relaciones para evitar duplicados.
- Concatena por columnas y filtra filas con centralidades todas en 0 (sin informacion relacional).
- Guarda el resultado en un CSV final.

In [7]:
def unificar_features_y_relaciones(
    features_path="large_twitch_features.csv",
    relaciones_path="datos_relacionales_20k.csv",
    salida_path="features_relaciones_20k.csv",
    limite=20000,
 ):
    features_df = pd.read_csv(features_path)
    relaciones_df = pd.read_csv(relaciones_path)

    if limite is not None:
        relaciones_df = relaciones_df.head(limite).reset_index(drop=True)

    if "nodo" in relaciones_df.columns:
        # Alinea por id real de nodo, no por indice
        unificado_df = features_df.merge(
            relaciones_df,
            left_on="numeric_id",
            right_on="nodo",
            how="inner",
        )
        unificado_df = unificado_df.drop(columns=["nodo"])
    else:
        unificado_df = pd.concat([features_df.head(limite).reset_index(drop=True), relaciones_df], axis=1)

    # Eliminar filas donde todas las centralidades sean 0 o NaN
    centralidades_cols = [
        "degree_centrality",
        "betweenness_centrality",
        "closeness_centrality",
    ]
    unificado_df = unificado_df[
        ((unificado_df[centralidades_cols] != 0) | unificado_df[centralidades_cols].isna()).any(axis=1)
    ]

    unificado_df.to_csv(salida_path, index=False)
    return unificado_df

# unificado_df = unificar_features_y_relaciones()
# unificado_df.head()

unificado_df_20k_to_40k = unificar_features_y_relaciones(
    features_path="large_twitch_features.csv",
    relaciones_path="datos_relacionales_20k_to_40k.csv",
    salida_path="features_relaciones_20k_to_40k.csv",
    limite=40000,
 )
unificado_df_20k_to_40k.head()

,views,mature,life_time,created_at,updated_at,numeric_id,dead_account,language,affiliate,degree_centrality,betweenness_centrality,closeness_centrality,clustering,triangles,pageRank
2,31839,1,1066,2015-11-11,2018-10-12,20002,0,EN,1,0.00035,2.177020e-04,0.240521,0.000000,0,0.000045
3,7729,0,1903,2013-07-13,2018-09-28,20003,0,EN,0,0.00015,3.740329e-07,0.232296,0.000000,0,0.000019
4,322,0,1994,2013-04-25,2018-10-10,20004,0,EN,0,0.00045,4.174369e-04,0.289590,0.111111,4,0.000061
5,2562,0,1926,2013-06-30,2018-10-08,20005,0,EN,1,0.00045,9.569587e-06,0.262475,0.055556,2,0.000044
6,19689,0,1603,2014-05-23,2018-10-12,20006,0,PL,0,0.00005,0.000000e+00,0.237446,0.000000,0,0.000012
